# Ma Soi — RL on Kaggle, headless (projects a / m)

Same experiment as section 6 of `train_bc_local.ipynb` — v3 run names and flags (spec
`docs/superpowers/specs/2026-09-17-rl-ppo-from-bc-design.md`, D11). Built for **"Save &
Run All"**: Kaggle runs it in the background, the browser can be closed and your machine
can be off. CPU sessions have **4 cores** and a **12 h** limit, so each version runs ONE
stage and stops itself at 11 h; the next version resumes where it stopped.

## One-time setup

1. **Kaggle account, phone-verified** (Settings → Phone verification). Needed to turn on
   Internet, which `npm ci` and the Node download require.
2. **Package the code** on your machine (only COMMITTED files go in). `apps/web/src` is
   left out: Kaggle rejects the `[` in Next.js route folders (`room/[code]`), and RL
   never needs the web app. `apps/web/package.json` stays, so `npm ci` still matches
   the lockfile (checked: `npm ci` + build + `rl_loop --help` pass on this archive):

   ```powershell
   git archive --format=zip HEAD -o .tmp\repo.zip -- . ":(exclude)apps/web/src"
   ```

3. **Dataset:** kaggle.com → Datasets → New Dataset → upload `repo.zip` (private).
   Kaggle may unzip it; this notebook handles both forms.
4. **Notebook:** Code → New Notebook → File → Import Notebook → this file.
   Right panel: **Add Input** → your dataset; **Settings**: Accelerator **None (CPU)**,
   **Internet ON**, Persistence off is fine.

## Each run (one stage)

1. Set `PROJECT` and `STAGE` in the next cell (project `m` starts with `village`).
2. **Add Input** → *Your Work* → this notebook's PREVIOUS version (skip on the very first run).
3. **Save Version** → **Save & Run All**. Close the browser if you like.
4. When it finishes, open the version's log: the last lines print `>>> NEXT:` — the stage for the next version.
5. After `confirm`, copy the `VERDICT` block (with the `tableSizes:` line) to Claude.

After a code change: re-run the same `git archive` command, upload a new dataset version, and the
notebook uses it automatically (it prints the commit).

In [ ]:
# ======================= EDIT THIS CELL, then "Save Version" → "Save & Run All" =======================
# PROJECT: "m" = bàn 8–12 người (spec 2026-09-22) | "a" = bàn 8 (spec 2026-09-17) | "b" = spec 2026-09-19 (đã đóng)
# One stage per version (the last cell prints which one to run next):
#   "village"      20 iterations, village side
#   "village-lr3"  only if "village" never promoted
#   "wolves"       20 iterations, wolves side, from the village champion
#   "wolves-lr3"   only if "wolves" never promoted
#   "confirm"      fresh-seed benchmark of the final candidate + VERDICT
#   "status"       print what is done, change nothing
PROJECT = "m"
STAGE = "village"

BUDGET_HOURS = 11.0   # Kaggle CPU sessions stop at 12 h; stop cleanly before that so the output is saved

## 0. Code, Node.js, dependencies

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path

T0 = time.time()
DEADLINE = T0 + BUDGET_HOURS * 3600

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")          # saved as this version's output
RL = WORK / "rl"                        # run folders live here (pruned after each iteration)
ROOT = Path("/tmp/repo")                # code + node_modules: NOT saved to output
TRAIN_DIR = ROOT / "ai-training"
NODE_VERSION = "v22.17.0"


def find_code():
    zips = [Path(p) for p in glob.glob(str(INPUT / "**" / "repo.zip"), recursive=True)]
    if zips:
        return "zip", zips[0]
    # Kaggle may auto-extract an uploaded zip: look for the repo root itself.
    for pkg in glob.glob(str(INPUT / "**" / "package.json"), recursive=True):
        root = Path(pkg).parent
        if (root / "ai-training" / "rl_loop.py").exists():
            return "dir", root
    raise SystemExit("repo not found under /kaggle/input - attach the dataset holding repo.zip (Step B)")


kind, src = find_code()
shutil.rmtree(ROOT, ignore_errors=True)
if kind == "zip":
    with zipfile.ZipFile(src) as archive:
        archive.extractall(ROOT)
        commit = archive.comment.decode() or "?"
else:
    shutil.copytree(src, ROOT)
    commit = "? (extracted dataset: no zip comment)"
print("code:", src, "| commit", commit)

# Node 22 from the official tarball (no apt / root needed).
node_dir = Path(f"/tmp/node-{NODE_VERSION}-linux-x64")
if not (node_dir / "bin" / "node").exists():
    url = f"https://nodejs.org/dist/{NODE_VERSION}/node-{NODE_VERSION}-linux-x64.tar.xz"
    try:
        tar_path, _ = urllib.request.urlretrieve(url, "/tmp/node.tar.xz")
    except OSError as error:
        raise SystemExit(f"cannot download Node ({error}). Settings -> Internet must be ON "
                         "(needs a phone-verified Kaggle account).")
    with tarfile.open(tar_path) as archive:
        archive.extractall("/tmp")
os.environ["PATH"] = f"{node_dir / 'bin'}:{os.environ['PATH']}"
print("node", subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip(),
      "| vCPU", os.cpu_count(), "| python", sys.version.split()[0])

done = subprocess.run("npm ci --no-audit --no-fund --loglevel=error && npm run build:deps --silent",
                      shell=True, cwd=ROOT)
if done.returncode != 0:
    raise SystemExit(f"npm ci / build failed (exit {done.returncode})")
import numpy
import torch
print("deps OK | torch", torch.__version__, "| numpy", numpy.__version__)

## 1. Restore the previous version's runs

In [ ]:
# Resume: "Add Input" -> "Your Work" -> this notebook's previous version. Its output
# (the rl/ folder) is copied back so rl_loop skips every finished step.
RL.mkdir(parents=True, exist_ok=True)
previous = sorted({Path(p).parent for p in glob.glob(str(INPUT / "**" / "rl" / "*" / "state.json"), recursive=True)})
for run in previous:
    dest = RL / run.name
    if not dest.exists():
        shutil.copytree(run, dest)
        print("restored", run.name, "from", run)
for log in glob.glob(str(INPUT / "**" / "rl" / "*.log"), recursive=True):
    if not (RL / Path(log).name).exists():
        shutil.copy2(log, RL / Path(log).name)
print("runs present:", sorted(p.name for p in RL.iterdir()) or "none (fresh start)")

## 2. Run the stage

In [ ]:
# Same runner as local (ai-training/rl_stages.py): same run folders, flags, resume and
# time budget. The remaining budget is passed down so the version ends before 12 h and
# Kaggle saves /kaggle/working (the rl/ folder) as this version's output.
remaining = max(0.2, (DEADLINE - time.time()) / 3600)
cmd = [sys.executable, str(TRAIN_DIR / "rl_stages.py"), "--project", PROJECT, STAGE,
       "--rl-dir", str(RL), "--budget-hours", f"{remaining:.2f}"]
print("$", " ".join(cmd), flush=True)
done = subprocess.run(cmd, cwd=str(ROOT), env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"})
print(f"\nexit {done.returncode} | elapsed {(time.time() - T0) / 3600:.1f} h")

## 3. Next

The last lines above print `>>> NEXT:`. Set `STAGE` to it, attach THIS version's output as input, and run a new version.